## Load useful libraries

In [1]:
import os
import subprocess
import glob
import json

from dotenv import set_key

import pprint as pp

## User settings

In [2]:
directory_openclaw_code = '/home/emily/Desktop/packages/openclaw'
directory_openclaw_config = '/home/emily/.openclaw'
directory_openclaw_auth = '/home/emily/.openclaw-auth-profile-secrets'
default_model = 'qwen3.6:35b'

## Initialize variables from environment

In [3]:
brave_api_key = os.environ['BRAVE_API_KEY']

## Replace the existing .env file with a copy of .env.example

In [4]:
cmd = ('cp ' + directory_openclaw_code + '/.env.example ' + directory_openclaw_code + '/.env').split(' ')
subprocess.run(cmd)

CompletedProcess(args=['cp', '/home/emily/Desktop/packages/openclaw/.env.example', '/home/emily/Desktop/packages/openclaw/.env'], returncode=0)

## Generate an OPENCLAW_GATEWAY_TOKEN and insert it into the .env file

In [5]:
token_process_result = subprocess.run(['openssl', 'rand', '-hex', '32'], capture_output=True, text=True)
openclaw_gateway_token = token_process_result.stdout.strip()
set_key(directory_openclaw_code + '/.env', 'OPENCLAW_GATEWAY_TOKEN', openclaw_gateway_token)

(True,
 'OPENCLAW_GATEWAY_TOKEN',
 'f0ec5f7d5a24f0883ddd9c0b096020084fe1014a3e3f5ddbf2f813678be32348')

## Construct the OpenClaw configuration dictionary

In [6]:
list_json_files = glob.glob('json/*.json')

In [7]:
dict_full = {}
for filename in list_json_files:
    key = filename.split('/')[-1].replace('.json', '')
    with open(filename) as f:
        dictionary = json.load(f)
        dict_full[key] = dictionary

## Ensure the model we intend to use as default is available to us

In [8]:
assert default_model in [x['id'] for x in dict_full['models']['providers']['ollama']['models']]

## Edit the default model

In [9]:
dict_full['agents']['defaults']['model']['primary'] = 'ollama' + '/' + default_model

In [10]:
dict_full['agents']['defaults']['models'] = {'ollama' + '/' + default_model : {}}

## Edit the Brave API key

In [11]:
dict_full['plugins']['entries']['brave']['config']['webSearch']['apiKey'] = brave_api_key

## Remove the previous configuration and auth directories

In [12]:
cmd = ('rm -R ' + directory_openclaw_config).split(' ')
subprocess.run(cmd)

CompletedProcess(args=['rm', '-R', '/home/emily/.openclaw'], returncode=0)

In [13]:
cmd = ('rm -R ' + directory_openclaw_auth).split(' ')
subprocess.run(cmd)

rm: cannot remove '/home/emily/.openclaw-auth-profile-secrets': No such file or directory


CompletedProcess(args=['rm', '-R', '/home/emily/.openclaw-auth-profile-secrets'], returncode=1)

## Create an empty configuration directory and workspace

In [14]:
cmd = ('mkdir ' + directory_openclaw_config).split(' ')
subprocess.run(cmd)

CompletedProcess(args=['mkdir', '/home/emily/.openclaw'], returncode=0)

In [15]:
cmd = ('mkdir ' + directory_openclaw_config + '/workspace').split(' ')
subprocess.run(cmd)

CompletedProcess(args=['mkdir', '/home/emily/.openclaw/workspace'], returncode=0)